# DocuCompare AI — Setup and Launch

Run cells **top to bottom** once. Then click the tunnel URL at the end.

In [ ]:
# CELL 1: Install all dependencies
!pip install -q streamlit scikit-learn gensim sentence-transformers transformers torch numpy pandas matplotlib pymupdf


In [ ]:
# CELL 2: Upload and extract your project zip
from google.colab import files
uploaded = files.upload()

import zipfile, os
with zipfile.ZipFile('docucompare.zip', 'r') as z:
    z.extractall('.')
os.chdir('docucompare')
print('Files found:', os.listdir('.'))


In [ ]:
# CELL 3: Launch Streamlit with cloudflared tunnel (no account needed)
import subprocess
import time

# Step 1: Download cloudflared binary
subprocess.run([
    'wget', '-q',
    'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
    '-O', 'cloudflared'
], check=True)
subprocess.run(['chmod', '+x', 'cloudflared'], check=True)
print('cloudflared ready')

# Step 2: Start Streamlit in background
streamlit_proc = subprocess.Popen(
    ['streamlit', 'run', 'app.py',
     '--server.port=8501',
     '--server.headless=true',
     '--server.enableCORS=false',
     '--server.enableXsrfProtection=false'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(6)
print('Streamlit started')

# Step 3: Open public tunnel
tunnel_proc = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://localhost:8501'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

print('Waiting for tunnel URL...')
for line in tunnel_proc.stdout:
    text = line.decode('utf-8').strip()
    if 'trycloudflare.com' in text:
        for part in text.split():
            if 'trycloudflare.com' in part:
                print('\n' + '='*55)
                print('DocuCompare AI is LIVE at:')
                print(part)
                print('='*55)
                print('Open the URL above in a new browser tab!')
                break
        break
